In [6]:
import pandas as pd
import numpy as np

print("--- Starting ETL Process ---")

# 1. EXTRACTION (All 9 Files)
df_orders = pd.read_csv('olist_orders_dataset.csv')
df_items = pd.read_csv('olist_order_items_dataset.csv')
df_products = pd.read_csv('olist_products_dataset.csv')
df_customers = pd.read_csv('olist_customers_dataset.csv')
df_sellers = pd.read_csv('olist_sellers_dataset.csv')
df_geo = pd.read_csv('olist_geolocation_dataset.csv')
df_translation = pd.read_json('category_catalog_api.json')
df_reviews = pd.read_csv('olist_order_reviews_dataset.csv')
df_payments = pd.read_csv('olist_order_payments_dataset.csv')

print("Extraction Complete.")

--- Starting ETL Process ---
Extraction Complete.


In [9]:
# 2. TRANSFORMATION (Cleaning & Generating Surrogate Keys)

# --- A. Date Cleaning ---
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'], errors='coerce')
df_orders['order_delivered_customer_date'] = pd.to_datetime(df_orders['order_delivered_customer_date'], errors='coerce')

# --- B. Dim_Location ---
dim_location = df_geo.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_city': 'first',
    'geolocation_state': 'first',
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean'
}).reset_index()
dim_location.rename(columns={'geolocation_zip_code_prefix': 'zip_code_prefix'}, inplace=True)
dim_location['Location_Key'] = range(1, len(dim_location) + 1) # Create Surrogate Key

# --- C. Dim_Product ---
df_products['product_weight_g'].fillna(df_products['product_weight_g'].median(), inplace=True)
dim_product = pd.merge(df_products, df_translation, on='product_category_name', how='left')
dim_product = dim_product[['product_id', 'product_category_name_english', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']]
dim_product['product_category_name_english'].fillna('Unknown', inplace=True)
dim_product['Product_Key'] = range(1, len(dim_product) + 1) # Create Surrogate Key

# --- D. Dim_Customer ---
dim_customer = df_customers[['customer_id', 'customer_unique_id']].drop_duplicates()
dim_customer['Customer_Key'] = range(1, len(dim_customer) + 1) # Create Surrogate Key

# --- E. Dim_Seller ---
dim_seller = df_sellers[['seller_id', 'seller_city', 'seller_state']].drop_duplicates()
dim_seller['Seller_Key'] = range(1, len(dim_seller) + 1) # Create Surrogate Key

# --- F. Dim_Date ---
dim_date = pd.DataFrame({'full_date': df_orders['order_purchase_timestamp'].dt.date.dropna().unique()})
dim_date['full_date'] = pd.to_datetime(dim_date['full_date'])
dim_date['Date_Key'] = dim_date['full_date'].dt.strftime('%Y%m%d').astype(int)
dim_date['year'] = dim_date['full_date'].dt.year
dim_date['quarter'] = dim_date['full_date'].dt.quarter
dim_date['month'] = dim_date['full_date'].dt.month
dim_date['day'] = dim_date['full_date'].dt.day
dim_date['day_of_week'] = dim_date['full_date'].dt.day_name()

# --- G. Clean Reviews & Payments ---
df_reviews_clean = df_reviews[['order_id', 'review_score']].drop_duplicates(subset=['order_id'])
df_payments_clean = df_payments[['order_id', 'payment_type']].drop_duplicates(subset=['order_id'])

# --- H. Fact Table (Merging Everything using Natural Keys, then keeping Surrogate Keys) ---
fact_sales = pd.merge(df_items, df_orders, on='order_id', how='inner')
fact_sales = pd.merge(fact_sales, dim_customer, on='customer_id', how='left')
fact_sales = pd.merge(fact_sales, dim_product, on='product_id', how='left')
fact_sales = pd.merge(fact_sales, dim_seller, on='seller_id', how='left')

# Map Location_Key using customer's zip code
fact_sales = pd.merge(fact_sales, df_customers[['customer_id', 'customer_zip_code_prefix']], on='customer_id', how='left')
fact_sales = pd.merge(fact_sales, dim_location[['zip_code_prefix', 'Location_Key']], left_on='customer_zip_code_prefix', right_on='zip_code_prefix', how='left')

# Add Reviews and Payments
fact_sales = pd.merge(fact_sales, df_reviews_clean, on='order_id', how='left')
fact_sales = pd.merge(fact_sales, df_payments_clean, on='order_id', how='left')

fact_sales['review_score'].fillna(0, inplace=True)
fact_sales['payment_type'].fillna('not_defined', inplace=True)

# Generate Date_Key for Fact Table
fact_sales['Date_Key'] = fact_sales['order_purchase_timestamp'].dt.strftime('%Y%m%d').fillna(0).astype(int)

# Derived Measures
fact_sales['total_order_value'] = fact_sales['price'] + fact_sales['freight_value']
fact_sales['delivery_duration_days'] = (fact_sales['order_delivered_customer_date'] - fact_sales['order_purchase_timestamp']).dt.days

# Keep ONLY the Star Schema Columns (Drop all old text IDs!)
fact_sales = fact_sales[['Date_Key', 'Customer_Key', 'Product_Key', 'Location_Key', 'Seller_Key', 'price', 'freight_value', 'total_order_value', 'delivery_duration_days', 'review_score', 'payment_type']]

# Drop rows where critical foreign keys couldn't be matched
fact_sales.dropna(subset=['Date_Key', 'Customer_Key', 'Product_Key', 'Location_Key', 'Seller_Key'], inplace=True)

# Convert Keys to Integer to prevent decimals in CSV
keys_cols = ['Date_Key', 'Customer_Key', 'Product_Key', 'Location_Key', 'Seller_Key']
fact_sales[keys_cols] = fact_sales[keys_cols].astype(int)

print("Transformation Complete.")

/tmp/ipykernel_2773/2242154003.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_products['product_weight_g'].fillna(df_products['product_weight_g'].median(), inplace=True)
/tmp/ipykernel_2773/2242154003.py:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when d

Transformation Complete.


In [10]:
# 3. LOADING (Exporting all 6 clean tables)
dim_location.to_csv('Clean_Dim_Location.csv', index=False)
dim_product.to_csv('Clean_Dim_Product.csv', index=False)
dim_customer.to_csv('Clean_Dim_Customer.csv', index=False)
dim_seller.to_csv('Clean_Dim_Seller.csv', index=False)
dim_date.to_csv('Clean_Dim_Date.csv', index=False)
fact_sales.to_csv('Clean_Fact_Sales.csv', index=False)

print("Data exported successfully! Please download ALL 6 Clean CSV files.")

Data exported successfully! Please download ALL 6 Clean CSV files.
